# NLP Task 5 — Fine-Tuning BERT for POS Tagging & Chunking

In [1]:
!pip install transformers datasets seqeval evaluate torch --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## Task 1: Dataset Selection

In [3]:
# BramVanroy/conll2003 is CoNLL-2003 in pure Parquet format — compatible with datasets >= 4.0
# (both 'conll2003' and 'eriktks/conll2003' still use deprecated loading scripts)
dataset = load_dataset('BramVanroy/conll2003')
print(dataset)
print('\nSample tokens :', dataset['train'][0]['tokens'])
print('POS tags       :', dataset['train'][0]['pos_tags'])
print('Chunk tags     :', dataset['train'][0]['chunk_tags'])

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Sample tokens : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS tags       : [22, 42, 16, 21, 35, 37, 16, 21, 7]
Chunk tags     : [11, 21, 11, 12, 21, 22, 11, 12, 0]


In [4]:
pos_label_list   = dataset['train'].features['pos_tags'].feature.names
chunk_label_list = dataset['train'].features['chunk_tags'].feature.names

print(f'POS labels   ({len(pos_label_list)}):', pos_label_list)
print(f'Chunk labels ({len(chunk_label_list)}):', chunk_label_list)

POS labels   (47): ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk labels (23): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


## Task 2: Data Preprocessing — Tokenization & Label Alignment

BERT uses WordPiece tokenization which splits words into subwords (e.g. `playing` → `play`, `##ing`). Since each word has one label but may produce multiple tokens, we assign the label to the first subword only and set `-100` for the rest so they are ignored by the loss function.

In [5]:
MODEL_NAME = 'bert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def align_labels(examples, label_col):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples[label_col]):
        word_ids     = tokenized.word_ids(batch_index=i)
        aligned      = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)              # special tokens [CLS], [SEP]
            elif word_id != prev_word_id:
                aligned.append(labels[word_id])   # first subword gets the real label
            else:
                aligned.append(-100)              # subsequent subwords ignored
            prev_word_id = word_id
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized

print('Tokenizer loaded.')

Tokenizer loaded.


In [6]:
pos_dataset = dataset.map(
    lambda x: align_labels(x, 'pos_tags'),
    batched=True,
    remove_columns=dataset['train'].column_names
)

chunk_dataset = dataset.map(
    lambda x: align_labels(x, 'chunk_tags'),
    batched=True,
    remove_columns=dataset['train'].column_names
)

# Verify label alignment on first sample
sample_ids    = pos_dataset['train'][0]['input_ids']
sample_labels = pos_dataset['train'][0]['labels']
tokens        = tokenizer.convert_ids_to_tokens(sample_ids)

print(f'{"Token":<15} Label')
for tok, lbl in zip(tokens[:12], sample_labels[:12]):
    print(f'{tok:<15} {lbl}')

Token           Label
[CLS]           -100
eu              22
rejects         42
german          16
call            21
to              35
boycott         37
british         16
lamb            21
.               7
[SEP]           -100


## Task 3: Model Setup

In [7]:
pos_id2label   = {i: l for i, l in enumerate(pos_label_list)}
pos_label2id   = {l: i for i, l in enumerate(pos_label_list)}
chunk_id2label = {i: l for i, l in enumerate(chunk_label_list)}
chunk_label2id = {l: i for i, l in enumerate(chunk_label_list)}

pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(pos_label_list),
    id2label=pos_id2label,
    label2id=pos_label2id
)

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print(f'POS model   — num_labels: {len(pos_label_list)}')
print(f'Chunk model — num_labels: {len(chunk_label_list)}')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 4367.87it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading 

POS model   — num_labels: 47
Chunk model — num_labels: 23


## Task 4: Training

In [8]:
!pip install "accelerate>=1.1.0" --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
seqeval       = evaluate.load('seqeval')
data_collator = DataCollatorForTokenClassification(tokenizer)

def make_compute_metrics(id2label):
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)
        true_labels = [
            [id2label[l] for l in row if l != -100]
            for row in labels
        ]
        true_preds = [
            [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]
        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            'precision': results['overall_precision'],
            'recall'   : results['overall_recall'],
            'f1'       : results['overall_f1'],
            'accuracy' : results['overall_accuracy']
        }
    return compute_metrics


def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy='epoch',        # renamed from evaluation_strategy
        save_strategy='epoch',
        load_best_model_at_end=True,
        logging_steps=50,
        report_to='none'
    )

print('Training setup ready.')

Training setup ready.


In [ ]:
pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args('./pos_model'),
    train_dataset=pos_dataset['train'],
    eval_dataset=pos_dataset['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_id2label)
)

pos_trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args('./chunk_model'),
    train_dataset=chunk_dataset['train'],
    eval_dataset=chunk_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_id2label)
)

chunk_trainer.train()

## Task 5: Evaluation using seqeval

In [ ]:
pos_results = pos_trainer.evaluate(pos_dataset['test'])

print('--- POS Tagging ---')
print(f'Precision : {pos_results["eval_precision"]:.4f}')
print(f'Recall    : {pos_results["eval_recall"]:.4f}')
print(f'F1 Score  : {pos_results["eval_f1"]:.4f}')
print(f'Accuracy  : {pos_results["eval_accuracy"]:.4f}')

In [ ]:
chunk_results = chunk_trainer.evaluate(chunk_dataset['test'])

print('--- Chunking ---')
print(f'Precision : {chunk_results["eval_precision"]:.4f}')
print(f'Recall    : {chunk_results["eval_recall"]:.4f}')
print(f'F1 Score  : {chunk_results["eval_f1"]:.4f}')
print(f'Accuracy  : {chunk_results["eval_accuracy"]:.4f}')

## Task 6: Inference on Custom Sentences

In [ ]:
def predict_tags(sentence, model, id2label):
    words  = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True
    ).to(device)

    model.to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids    = inputs.word_ids()

    # Take prediction from first subword only for each word
    word_preds = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in word_preds:
            word_preds[word_id] = id2label[predictions[idx]]

    return list(zip(words, [word_preds[i] for i in range(len(words))]))

In [ ]:
# Example from assignment
sentence = 'John works at Google in California'

pos_out   = predict_tags(sentence, pos_model,   pos_id2label)
chunk_out = predict_tags(sentence, chunk_model, chunk_id2label)

print(f'Input: {sentence}\n')
print(f'{"Word":<15} {"POS Tag":<12} Chunk Tag')
print('-' * 42)
for (word, pos), (_, chunk) in zip(pos_out, chunk_out):
    print(f'{word:<15} {pos:<12} {chunk}')

In [ ]:
# More inference examples
sentences = [
    'The quick brown fox jumps over the lazy dog',
    'Anthropic released a new AI model last week',
    'She sold seashells by the seashore'
]

for sent in sentences:
    pos_out   = predict_tags(sent, pos_model,   pos_id2label)
    chunk_out = predict_tags(sent, chunk_model, chunk_id2label)
    print(f'\nInput: {sent}')
    print(f'{"Word":<22} {"POS":<12} Chunk')
    print('-' * 48)
    for (word, pos), (_, chunk) in zip(pos_out, chunk_out):
        print(f'{word:<22} {pos:<12} {chunk}')

## Task 7: Comparison — POS Tagging vs Chunking

In [ ]:
comparison = pd.DataFrame({
    'Metric'      : ['Precision', 'Recall', 'F1 Score', 'Accuracy'],
    'POS Tagging' : [round(pos_results[f'eval_{k}'], 4)   for k in ['precision','recall','f1','accuracy']],
    'Chunking'    : [round(chunk_results[f'eval_{k}'], 4) for k in ['precision','recall','f1','accuracy']]
})
print(comparison.to_string(index=False))

In [ ]:
metrics    = ['Precision', 'Recall', 'F1 Score', 'Accuracy']
pos_vals   = [pos_results[f'eval_{k}']   for k in ['precision','recall','f1','accuracy']]
chunk_vals = [chunk_results[f'eval_{k}'] for k in ['precision','recall','f1','accuracy']]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, pos_vals,   width, label='POS Tagging', color='steelblue',  alpha=0.85)
b2 = ax.bar(x + width/2, chunk_vals, width, label='Chunking',    color='darkorange', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title('POS Tagging vs Chunking — Performance Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## Task 8: Report

### Differences between POS Tagging and Chunking

POS tagging assigns a grammatical category to each individual word — noun, verb, adjective, preposition, and so on. It works purely at the word level. Chunking goes one step further and groups words into meaningful phrases — noun phrases (NP), verb phrases (VP), prepositional phrases (PP) — using BIO notation where `B-NP` marks the start of a noun phrase, `I-NP` continues it, and `O` marks tokens outside any phrase. In short: POS tells you what each word *is*, chunking tells you how words *group together*.

### Challenges Faced

The most significant challenge was label alignment with BERT's WordPiece tokenizer. Words like `playing` get split into `play` and `##ing`, but there is only one label per word. Assigning `-100` to all subword continuations and special tokens (`[CLS]`, `[SEP]`) was essential so the loss function ignores them. Without this, the model would try to learn incorrect boundaries.

Chunking was harder than POS tagging because of BIO notation — a wrong `B` tag corrupts the entire phrase boundary even if the remaining `I` tags are correct. The `seqeval` metric evaluates at the chunk level, so a single boundary error penalises the whole phrase.

### Observations and Insights

POS tagging achieves higher scores than chunking, which is expected given the simpler label structure. BERT handles both well because its bidirectional self-attention captures context from both directions — this is especially useful for resolving words with multiple roles (e.g. `bank` as noun vs. verb). The `seqeval` F1 being lower for chunking despite similar token-level accuracy confirms that phrase-boundary detection is the harder sub-problem.